In [2]:
%run run_all_subjects.py --seed 999 --adabn

[SEED CONTROL] Using seed=999 (numpy/torch/python random all seeded, cuDNN deterministic)
NOTE: tsfnet_model package not found; loading definitions from E:\kntu\Thesis\codes\HOPE-LAST-CODE-FILE\new-tsfnet\model\G4\tsfnet_model_notebook.py instead.
[RESULTS] Per-subject session-level JSON/checkpoints for this run go to: E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999
===== CrashSafeLogger started [11SEG_BASELINE_PAPER_MEAN_NOCLIP_NOCLASSWEIGHT_NOSMOTE_GAP_CAFE64_NOASYM_ADABN] 2026-09-12 22:36:12 -> E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\baseline_run_log_20260912_223612_seed999_fda0fd34fd.txt =====
===== FULL BASELINE RUN started 20260912_223612_seed999_fda0fd34fd (seed=999) =====
[ABLATION] asymmetric_pairs=None weight=2.0 ordinal_reg_lambda=0.0 (priority-A1 isolated variable) | lambda_eeg=0.2 lambda_fnirs=0.2 (priority-A3 isolated variable, paper default 0.2/0.2) | cafe_d_model=64 (priority-4 result: rejected) |

In [4]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [5]:
!pip install shap --break-system-packages -q

import os, torch
from tsfnet_model_notebook import TSFNet, NBackDataset, _unpack_batch
from torch.utils.data import DataLoader
from xai_extensions import (
    run_shap_analysis, plot_fusion_gating_weights,
    plot_efgf_attention_map, run_cafe_attention_analysis,
    run_gradcam_analysis, run_group_level_tsne, run_single_subject_tsne,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = r"مسیر پوشه‌ی نتایج اجرای E3+E4 (همون که run34 توش نوشت)"
SEED = 999          # کدوم بذر رو برای XAI انتخاب می‌کنید (پیشنهاد: بهترین/میانه)
TEST_SESSION = 1    # کدوم session به‌عنوان تست نگه داشته شده


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
def load_subject_model_and_loader(subject_id, subject_dir):
    ckpt_path = os.path.join(RESULTS_DIR, f"{subject_id}_session{TEST_SESSION}_best.pt")
    model = TSFNet(num_classes=3, cafe_d_model=64, fnirs_num_segments=21).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    full = NBackDataset(subject_dir, subject_id)
    test_idx = [i for i, s in enumerate(full.session_ids) if s == TEST_SESSION]
    train_idx = [i for i, s in enumerate(full.session_ids) if s != TEST_SESSION]
    test_ds = NBackDataset(subject_dir, subject_id, indices=test_idx)
    train_ds = NBackDataset(subject_dir, subject_id, indices=train_idx)
    test_loader = DataLoader(test_ds, batch_size=16)
    train_loader = DataLoader(train_ds, batch_size=16)
    return model, train_loader, test_loader

In [ ]:
import glob
# جست‌وجو برای هر پوشه‌ای که دقیقاً VP001_session1_best.pt رو داشته باشه
matches = glob.glob(r"C:\**\VP001_session1_best.pt", recursive=True)
print(matches)

In [1]:
SUBJECT_DIR_ROOT = r"E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback"
RESULTS_DIR = r"E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999"

In [12]:
SUBJECT_DIR_ROOT = r"E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback"
RESULTS_DIR = r"E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999"
subject_ids = [f"VP{i:03d}" for i in range(1, 27)]

for vp in subject_ids:
    subject_dir = os.path.join(SUBJECT_DIR_ROOT, vp)
    model, train_loader, test_loader = load_subject_model_and_loader(vp, subject_dir)
    out_dir = os.path.join(RESULTS_DIR, "xai_outputs", vp)

    run_gradcam_analysis(model, test_loader, device, TEST_SESSION, out_dir, n_samples=200)
    plot_fusion_gating_weights(model, out_dir, session_id=TEST_SESSION)
    plot_efgf_attention_map(model, test_loader, device, TEST_SESSION, out_dir)
    run_cafe_attention_analysis(model, test_loader, device, TEST_SESSION, out_dir)

    del model
    torch.cuda.empty_cache()

  [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP001\S1_gradcam_eeg_spatial.png
  [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP001\S1_gradcam_fnirs_spatial.png
  [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP001\S1_gradcam_fusion_spatial.png
  [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP001\S1_gradcam_eeg_temporal.png
  [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP001\S1_gradcam_fusion_temporal.png

  Grad-CAM analysis complete -> E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP001

  [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP001\S1_fusion_gating_weights.png  (w_eeg=0.739, w_fusion=0.724, w_fnirs=

In [15]:
import importlib
import xai_extensions
importlib.reload(xai_extensions)
from xai_extensions import (
    run_shap_analysis, plot_fusion_gating_weights,
    plot_efgf_attention_map, run_cafe_attention_analysis,
    run_gradcam_analysis, run_group_level_tsne, run_single_subject_tsne,
)

In [16]:
representative = ["VP017", "VP006", "VP013", "VP022", "VP009"]
for vp in representative:
    subject_dir = os.path.join(SUBJECT_DIR_ROOT, vp)
    model, _, test_loader = load_subject_model_and_loader(vp, subject_dir)
    out_dir = os.path.join(RESULTS_DIR, "xai_outputs", vp)
    run_shap_analysis(model, test_loader, device, TEST_SESSION, out_dir,
                       n_background=32, n_explain=64)
    del model 
    torch.cuda.empty_cache()

  [SHAP] background=32 samples, explain=64 samples ...
    [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP017\S1_shap_eeg_spatial.png
    [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP017\S1_shap_eeg_temporal.png
    [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP017\S1_shap_fnirs_segments.png
  [SHAP] saved raw values + plots -> E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP017
  [SHAP] background=32 samples, explain=64 samples ...
    [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP006\S1_shap_eeg_spatial.png
    [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP006\S1_shap_eeg_temporal.png
    [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_

In [21]:
import importlib
import xai_extensions
importlib.reload(xai_extensions)
from xai_extensions import (
    run_shap_analysis, plot_fusion_gating_weights,
    plot_efgf_attention_map, run_cafe_attention_analysis,
    run_gradcam_analysis, run_group_level_tsne, run_single_subject_tsne,
)

In [22]:
model, train_loader, test_loader = load_subject_model_and_loader("VP017", os.path.join(SUBJECT_DIR_ROOT, "VP017"))
run_single_subject_tsne(model, train_loader, test_loader, device,
                         "VP017", TEST_SESSION, os.path.join(RESULTS_DIR, "xai_outputs", "VP017"))

  [single-subject t-SNE] VP017, session 1 ...
    [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999\xai_outputs\VP017\VP017_S1_single_subject_tsne.png


In [1]:

import os, torch
from torch.utils.data import DataLoader
from tsfnet_model_notebook import TSFNet, NBackDataset, _unpack_batch
from xai_extensions import (
    run_shap_analysis, plot_fusion_gating_weights,
    plot_efgf_attention_map, run_cafe_attention_analysis,
    run_gradcam_analysis, run_group_level_tsne, run_single_subject_tsne,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = r"E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999"
SUBJECT_DIR_ROOT = r"E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback"
SEED = 999
TEST_SESSION = 1
subject_ids = [f"VP{i:03d}" for i in range(1, 27)]

In [2]:
subject_checkpoints = {vp: os.path.join(RESULTS_DIR, f"{vp}_session{TEST_SESSION}_best.pt") for vp in subject_ids}

def build_test_loader_only(subject_id):
    subject_dir = os.path.join(SUBJECT_DIR_ROOT, subject_id)
    full = NBackDataset(subject_dir, subject_id)
    test_idx = [i for i, s in enumerate(full.session_ids) if s == TEST_SESSION]
    test_ds = NBackDataset(subject_dir, subject_id, indices=test_idx)
    del full
    return DataLoader(test_ds, batch_size=16)

run_group_level_tsne(
    model_builder=lambda: TSFNet(num_classes=3, cafe_d_model=64, fnirs_num_segments=21),
    subject_checkpoints=subject_checkpoints,
    loader_builder=build_test_loader_only,
    device=device,
    output_dir=os.path.join(RESULTS_DIR, "xai_outputs", "group_level"),
)

  [group t-SNE] aggregating 26 subject/session entries ...
    [1/26] VP001: 299 windows
    [2/26] VP002: 315 windows
    [3/26] VP003: 315 windows
    [4/26] VP004: 252 windows
    [5/26] VP005: 314 windows
    [6/26] VP006: 305 windows
    [7/26] VP007: 295 windows
    [8/26] VP008: 294 windows
    [9/26] VP009: 311 windows
    [10/26] VP010: 302 windows
    [11/26] VP011: 309 windows
    [12/26] VP012: 305 windows
    [13/26] VP013: 299 windows
    [14/26] VP014: 314 windows
    [15/26] VP015: 302 windows
    [16/26] VP016: 305 windows
    [17/26] VP017: 314 windows
    [18/26] VP018: 290 windows
    [19/26] VP019: 287 windows
    [20/26] VP020: 268 windows
    [21/26] VP021: 253 windows
    [22/26] VP022: 301 windows
    [23/26] VP023: 315 windows
    [24/26] VP024: 306 windows
    [25/26] VP025: 315 windows
    [26/26] VP026: 315 windows
    t-SNE (eeg, 6000 windows from 26 subjects, perplexity=30) ... done
    [saved] E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results